# V3 Phase 2 — QA du dataset synthétique

20 scènes random avec overlay des annotations coins.

- **Boîte verte** : coin visible (`is_occluded=0`)
- **Boîte orange** : coin partiellement occlus (`is_occluded=1`)
- **Point rouge** : centre du digit (annotation `corner_x, corner_y`)
- La boîte = fenêtre `crop_size` que verra le corner-classifier (Phase 4)

Annotation : `<class> <corner_x> <corner_y> <is_occluded> <crop_size>`

In [ ]:
import random
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
IMG = ROOT / 'data' / 'synthetic' / 'images'
LBL = ROOT / 'data' / 'synthetic' / 'labels'
ids = sorted(p.stem for p in IMG.glob('*.jpg'))
print(f'{len(ids)} scenes disponibles')
random.seed(42)
sample = random.sample(ids, min(20, len(ids)))

In [ ]:
def draw(sid):
    img = cv2.imread(str(IMG / f'{sid}.jpg'))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    txt = (LBL / f'{sid}.txt').read_text().strip()
    nv = no = 0
    for line in txt.split('\n'):
        if not line.strip():
            continue
        p = line.split()
        lbl, x, y, o = p[0], int(p[1]), int(p[2]), int(p[3])
        cs = int(p[4]) if len(p) > 4 else 50
        h = cs // 2
        col = (255, 140, 0) if o else (0, 200, 0)
        no += o; nv += 1
        cv2.rectangle(img, (x-h, y-h), (x+h, y+h), col, 2)
        cv2.circle(img, (x, y), 3, (255, 0, 0), -1)
        cv2.putText(img, lbl, (x-h, y-h-4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1)
    return img, nv, no

fig, axes = plt.subplots(5, 4, figsize=(22, 18))
for ax, sid in zip(axes.flat, sample):
    im, nv, no = draw(sid)
    ax.imshow(im); ax.set_title(f'{sid} | {nv} coins, {no} occlus', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## Statistiques du dataset

In [ ]:
from collections import Counter
n_corners, n_occ = 0, 0
per_class = Counter()
per_scene = []
for sid in ids:
    lines = (LBL / f'{sid}.txt').read_text().strip().split('\n')
    lines = [l for l in lines if l.strip()]
    per_scene.append(len(lines))
    for l in lines:
        p = l.split(); per_class[p[0]] += 1
        n_corners += 1; n_occ += int(p[3])
print(f'Scenes      : {len(ids)}')
print(f'Coins total : {n_corners}  (moy {n_corners/len(ids):.1f}/scene)')
print(f'Occlus      : {n_occ} ({100*n_occ/max(n_corners,1):.1f}%)')
print(f'Classes     : {len(per_class)}/54')
print(f'min/max par classe : {min(per_class.values())} / {max(per_class.values())}')

fig, ax = plt.subplots(1, 2, figsize=(16, 4))
ax[0].hist(per_scene, bins=20, color='#3498db'); ax[0].set_title('Coins par scene')
cc = sorted(per_class.items())
ax[1].bar([k for k,_ in cc], [v for _,v in cc]); ax[1].set_title('Coins par classe')
ax[1].tick_params(axis='x', rotation=90, labelsize=6)
plt.tight_layout(); plt.show()